# PRDetect - Test GCN & RGCN trên toàn bộ tập dữ liệu
Notebook gọi trực tiếp các script sẵn có: `building_graph.py`, `test.py`

**Workflow mỗi tập:** build graph → test GCN → test RGCN → xóa `.pkl` → lưu kết quả

In [ ]:
# ============================================================
# CELL 1: Copy model weights từ Kaggle dataset input
# ============================================================
import os

WORK_DIR = '/kaggle/working/PRDetect-Research'
os.chdir(WORK_DIR)

os.makedirs('./model', exist_ok=True)

# Copy RGCN model
!cp /kaggle/input/datasets/brokensw1402/trongsohc3/hc3_rgcn_model_2026.pth \
   ./model/hc3_rgcn_model_2026.pth

# Copy GCN model (điều chỉnh đường dẫn nếu khác)
!cp /kaggle/input/datasets/brokensw1402/trongsohc3/hc3_gcn_model_2026.pth \
   ./model/hc3_gcn_model_2026.pth

print('[OK] Đã copy model weights.')
!ls -lh ./model/

In [ ]:
# ============================================================
# CELL 2: Danh sách toàn bộ file test cần chạy
# (Bỏ qua train / val)
# ============================================================
import os

WORK_DIR = '/kaggle/working/PRDetect-Research'

def is_test_file(fname):
    name = fname.lower()
    if not name.endswith('.json'):
        return False
    if 'train' in name or '_val' in name:
        return False
    skip_exact = {
        'gpt3.5_mixed_test_split.json',
        'matched_gpt3.5_mixed_test_split.json',
    }
    if name in skip_exact:
        return False
    return True

original_dir  = os.path.join(WORK_DIR, 'data/original_text')
perturbed_dir = os.path.join(WORK_DIR, 'data/perturbed_text')

original_files = [
    ('original', f[:-5])
    for f in sorted(os.listdir(original_dir))
    if is_test_file(f)
]
perturbed_files = [
    ('perturbed', f[:-5])
    for f in sorted(os.listdir(perturbed_dir))
    if is_test_file(f)
]

all_test_files = original_files + perturbed_files

print(f'Tổng số file sẽ test: {len(all_test_files)}')
print(f'  - data/original_text : {len(original_files)} file')
print(f'  - data/perturbed_text: {len(perturbed_files)} file')
print()
print('=== DANH SÁCH FILE ORIGINAL_TEXT ===')
for _, name in original_files:
    print(f'  {name}')
print()
print('=== DANH SÁCH FILE PERTURBED_TEXT ===')
for _, name in perturbed_files:
    print(f'  {name}')

In [ ]:
# ============================================================
# CELL 3: Vòng lặp test toàn bộ dataset
#
# Workflow mỗi tập:
#   1. python building_graph.py --file <name>  → tạo graph_data/<name>.pkl
#   2. python test.py --file <name> --model_type gcn  --dataset hc3 --seed 2026 -s
#   3. python test.py --file <name> --model_type rgcn --dataset hc3 --seed 2026 -s
#   4. Xóa file .pkl để giải phóng RAM
# Kết quả (-s) ghi vào test_result.txt sau mỗi lần chạy
# ============================================================
import os
import subprocess
from datetime import datetime

WORK_DIR     = '/kaggle/working/PRDetect-Research'
DATASET      = 'hc3'
SEED         = '2026'
BATCH_SIZE   = '16'   # nhỏ để tiết kiệm RAM Kaggle
RESULT_FILE  = os.path.join(WORK_DIR, 'RGCN_test.txt')

os.chdir(WORK_DIR)
os.makedirs('./graph_data', exist_ok=True)
os.makedirs('./result', exist_ok=True)

# Xóa/khởi tạo file kết quả cũ
with open(RESULT_FILE, 'w', encoding='utf-8') as f:
    f.write(f'=== PRDetect RGCN_test Results ===\n')
    f.write(f'Date: {datetime.now()} | Dataset: {DATASET} | Seed: {SEED}\n')
    f.write('=' * 80 + '\n')

failed = []

for idx, (src, file_name) in enumerate(all_test_files):
    print(f'\n[{idx+1}/{len(all_test_files)}] ► {file_name}  ({src})')
    print('─' * 65)

    graph_pkl = f'./graph_data/{file_name}.pkl'

    # ── Bước 1: Build graph ──────────────────────────────────
    if os.path.exists(graph_pkl):
        print(f'  [Cache] Sử dụng cache {graph_pkl}')
    else:
        print('  [Build] Đang build graph...')
        ret = subprocess.run(
            ['python', 'building_graph.py', '--file', file_name],
            capture_output=True, text=True
        )
        print(ret.stdout[-2000:] if ret.stdout else '')
        if ret.returncode != 0:
            print(f'  [ERROR] Build graph thất bại:\n{ret.stderr[-1000:]}')
            failed.append((file_name, 'build_graph'))
            continue
        print('  [OK] Build xong!')

    # ── Bước 2: Test GCN ─────────────────────────────────────
    print('  [GCN]  Đang test...')
    ret_gcn = subprocess.run(
        ['python', 'test.py',
         '--file', file_name,
         '--dataset', DATASET,
         '--seed', SEED,
         '--model_type', 'gcn',
         '--batch_size', BATCH_SIZE,
         '-s'],
        capture_output=True, text=True
    )
    gcn_out = ret_gcn.stdout
    print(gcn_out[-1500:] if gcn_out else '')
    if ret_gcn.returncode != 0:
        print(f'  [ERROR] GCN test lỗi:\n{ret_gcn.stderr[-500:]}')
        failed.append((file_name, 'gcn_test'))

    # ── Bước 3: Test RGCN ────────────────────────────────────
    print('  [RGCN] Đang test...')
    ret_rgcn = subprocess.run(
        ['python', 'test.py',
         '--file', file_name,
         '--dataset', DATASET,
         '--seed', SEED,
         '--model_type', 'rgcn',
         '--batch_size', BATCH_SIZE,
         '-s'],
        capture_output=True, text=True
    )
    rgcn_out = ret_rgcn.stdout
    print(rgcn_out[-1500:] if rgcn_out else '')
    if ret_rgcn.returncode != 0:
        print(f'  [ERROR] RGCN test lỗi:\n{ret_rgcn.stderr[-500:]}')
        failed.append((file_name, 'rgcn_test'))

    # ── Bước 4: Xóa .pkl giải phóng RAM ──────────────────────
    if os.path.exists(graph_pkl):
        os.remove(graph_pkl)
        print(f'  [MEM]  Đã xóa {graph_pkl}')

    print(f'  [DONE] ✓ {file_name}')

# ── Sau khi xong: copy test_result.txt → RGCN_test.txt ──────
src_result = './test_result.txt'
if os.path.exists(src_result):
    with open(src_result, 'r', encoding='utf-8') as f:
        content = f.read()
    with open(RESULT_FILE, 'a', encoding='utf-8') as f:
        f.write(content)

with open(RESULT_FILE, 'a', encoding='utf-8') as f:
    f.write('\n' + '=' * 80 + '\n')
    f.write(f'Hoàn thành: {datetime.now()}\n')
    f.write(f'Tổng: {len(all_test_files)} file | Lỗi: {len(failed)}\n')
    for fn, step in failed:
        f.write(f'  FAIL: {fn} ({step})\n')

print(f'\n[DONE] Kết quả lưu tại: {RESULT_FILE}')
if failed:
    print(f'[WARN] {len(failed)} file thất bại: {failed}')

In [ ]:
# ============================================================
# CELL 4: Hiển thị & so sánh kết quả GCN vs RGCN
# ============================================================
import re
from collections import defaultdict

RESULT_FILE = '/kaggle/working/PRDetect-Research/RGCN_test.txt'

# ── Đọc và parse RGCN_test.txt (format từ test.py -s) ──────
# Format: {file}\t acc: {acc}\t auc: {auc}\t f1: {f1}\t seed: {seed}\t model: {dataset}_{model_type}\t{time}
pattern = re.compile(
    r'^(\S+)\s+acc:\s+([\d.]+)\s+auc:\s+([\d.]+)\s+f1:\s+([\d.]+)'
    r'\s+seed:\s+(\S+)\s+model:\s+(\S+)'
)

records = defaultdict(dict)

with open(RESULT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        m = pattern.match(line.strip())
        if not m:
            continue
        fname, acc, auc, f1, seed, model_str = m.groups()
        acc, auc, f1 = float(acc), float(auc), float(f1)
        # model_str dạng "hc3_gcn" hoặc "hc3_rgcn"
        model_type = 'rgcn' if 'rgcn' in model_str else 'gcn'
        records[fname][model_type] = {'acc': acc, 'auc': auc, 'f1': f1}

def fmt(v):
    return f'{v:.4f}' if v is not None else '  N/A '

def delta(gv, rv):
    if gv is None or rv is None: return '   ---'
    d = rv - gv
    sign = '+' if d >= 0 else ''
    return f'{sign}{d:.4f}'

SEP = '─' * 130
print('=' * 130)
print(' PRDetect — SO SÁNH GCN vs RGCN | dataset=hc3 | seed=2026')
print('=' * 130)
print(f'{"FILE":<50} | {"GCN_ACC":>8} {"GCN_AUC":>8} {"GCN_F1":>7}'
      f' | {"RGCN_ACC":>9} {"RGCN_AUC":>9} {"RGCN_F1":>8}'
      f' | {"Δ_ACC":>8} {"Δ_AUC":>8} {"Δ_F1":>8}')
print(SEP)

gcn_wins = rgcn_wins = ties = {m: 0 for m in ['acc','auc','f1']}
gcn_wins  = defaultdict(int)
rgcn_wins = defaultdict(int)
ties      = defaultdict(int)

for fname in sorted(records.keys()):
    gcn  = records[fname].get('gcn')
    rgcn = records[fname].get('rgcn')
    g_acc = gcn['acc']  if gcn  else None
    g_auc = gcn['auc']  if gcn  else None
    g_f1  = gcn['f1']   if gcn  else None
    r_acc = rgcn['acc'] if rgcn else None
    r_auc = rgcn['auc'] if rgcn else None
    r_f1  = rgcn['f1']  if rgcn else None

    for metric, gv, rv in [('acc',g_acc,r_acc),('auc',g_auc,r_auc),('f1',g_f1,r_f1)]:
        if gv is not None and rv is not None:
            if rv > gv + 1e-4:   rgcn_wins[metric] += 1
            elif gv > rv + 1e-4: gcn_wins[metric]  += 1
            else:                ties[metric]       += 1

    print(f'{fname:<50} | {fmt(g_acc):>8} {fmt(g_auc):>8} {fmt(g_f1):>7}'
          f' | {fmt(r_acc):>9} {fmt(r_auc):>9} {fmt(r_f1):>8}'
          f' | {delta(g_acc,r_acc):>8} {delta(g_auc,r_auc):>8} {delta(g_f1,r_f1):>8}')

total = len(records)
print('\n' + '=' * 130)
print('TỔNG HỢP THẮNG/HÒA/THUA (RGCN so với GCN):')
print(f'  {"Metric":<6} | {"RGCN thắng":>10} | {"Hòa":>5} | {"GCN thắng":>9} | {"Tổng":>5}')
for m in ['acc','auc','f1']:
    print(f'  {m.upper():<6} | {rgcn_wins[m]:>10} | {ties[m]:>5} | {gcn_wins[m]:>9} | {total:>5}')

# Trung bình
def avg(model_key, metric):
    vals = [records[f][model_key][metric] for f in records if model_key in records[f]]
    return sum(vals)/len(vals) if vals else 0.0

print('\nTRUNG BÌNH TOÀN BỘ TẬP TEST:')
print(f'  {"":8} {"ACC":>8} {"AUC":>8} {"F1":>8}')
for mk, label in [('gcn','GCN'),('rgcn','RGCN')]:
    print(f'  {label:<8} {avg(mk,"acc"):>8.4f} {avg(mk,"auc"):>8.4f} {avg(mk,"f1"):>8.4f}')
print('=' * 130)

# In toàn bộ file
print(f'\n\n══════ NỘI DUNG FILE RGCN_test.txt ══════\n')
with open(RESULT_FILE, 'r', encoding='utf-8') as f:
    print(f.read())